# Selection fidelity and accuracy validation

This notebook runs seeds 42, 43, and 44 in one Colab session.

For fixed four-layer candidate groups at 50% pruning, it compares:

1. additive selection: sum of single-layer distortions;
2. pairwise-interaction-corrected selection: single-layer sum plus measured pair interactions;
3. direct-joint selection: measured joint distortion on the calibration split.

Each selected group is evaluated on a held-out test split, then fine-tuned. The notebook reports distortion prediction error, rank correlation, selection regret, immediate accuracy, and fine-tuned accuracy.

## Requirements

- Colab GPU runtime.
- Drive mounted with CIFAR-10 at `/content/drive/MyDrive/cifar10/cifar-10-batches-py`.
- Dense checkpoints at `/content/drive/MyDrive/rdhp_rdp_runs_seed_{42,43,44}/dense_resnet32.pt`.
- Colab Secret `GITHUB_TOKEN` with read access to the private `my-thesis` repository and its submodules.
- Colab Secret `GITHUB_WRITE_TOKEN` with Contents read/write access to this experiment repository.

The final publisher cell is enabled by default.

In [ ]:
from pathlib import Path
import copy
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
from argparse import Namespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from google.colab import drive

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required.')

askpass = Path('/tmp/github-askpass-selection-fidelity.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\\n' x-access-token ;;\n"
    "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(
            ['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)],
            env=git_env, check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'],
            env=git_env, check=True,
        )
finally:
    askpass.unlink(missing_ok=True)

EXPERIMENTS = REPO / 'working' / 'experiments'
sys.path.insert(0, str(EXPERIMENTS))

from rdp_resnet32_adapter import (
    accuracy,
    apply_masks,
    load_checkpoint,
    load_resnet32,
    logits,
    make_loaders,
    mask_topk,
    model_stats,
    prunable_layers,
    seed_everything,
    train_epochs,
)

DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    drive.mount('/content/drive')

DATA_ROOT = DRIVE_ROOT / 'cifar10'
DATASET_DIR = DATA_ROOT / 'cifar-10-batches-py'
for name in ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch']:
    if not (DATASET_DIR / name).exists():
        raise FileNotFoundError(f'Missing CIFAR-10 file: {DATASET_DIR / name}')

SEEDS = [42, 43, 44]
SELECTION_SEED = 123
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
PRUNE_RATIO = 0.50
LAYER_COUNT = 12
GROUP_SIZE = 4
CANDIDATE_GROUP_COUNT = 12
FINETUNE_EPOCHS = 5
LEARNING_RATE = 0.1
RUN_FINETUNE = True
PUBLISH_RESULTS = True

OUTPUT_ROOT = DRIVE_ROOT / 'rdhp_rdp_selection_fidelity_all_seeds'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Seeds:', SEEDS)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Output:', OUTPUT_ROOT)

In [ ]:
probe = load_resnet32()
all_layer_names = [
    name for name, module in probe.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]
del probe

selected_layers = np.linspace(
    0, len(all_layer_names) - 1,
    min(LAYER_COUNT, len(all_layer_names)),
    dtype=int,
).tolist()
selected_layers = list(dict.fromkeys(int(i) for i in selected_layers))

rng = np.random.default_rng(SELECTION_SEED)
candidate_groups = []
seen_groups = set()
while len(candidate_groups) < min(CANDIDATE_GROUP_COUNT, 100):
    group = tuple(sorted(int(x) for x in rng.choice(
        selected_layers, size=min(GROUP_SIZE, len(selected_layers)), replace=False
    )))
    if group not in seen_groups:
        seen_groups.add(group)
        candidate_groups.append(group)

print('Selected layers:')
print([(i, all_layer_names[i]) for i in selected_layers])
print('Candidate groups:')
for group_id, group in enumerate(candidate_groups):
    print(group_id, group, [all_layer_names[i] for i in group])

## Selection definitions

For each candidate group (G), let (d_i) be the single-layer distortion and (d_{ij}) the measured pair distortion.

- Additive: (S_\mathrm{add}(G)=\sum_i d_i)
- Pair-corrected: (S_\mathrm{pair}(G)=\sum_i d_i+\sum_{i<j}(d_{ij}-d_i-d_j))
- Direct joint: (S_\mathrm{joint}(G)=d_G)

All selections use calibration distortion. The selected group is then evaluated on the test split. The test oracle is reported only as a regret reference; it is not used for selection.

In [ ]:
def all_one_masks(model):
    return [
        torch.ones_like(layer.weight, dtype=torch.bool).cpu()
        for layer in prunable_layers(model)
    ]

def masks_for_group(model, group, ratio=PRUNE_RATIO):
    masks = all_one_masks(model)
    layers = prunable_layers(model)
    for layer_index in group:
        masks[int(layer_index)] = mask_topk(
            layers[int(layer_index)].weight,
            float(ratio),
        ).detach().cpu()
    return masks

def measure_distortion(model, masks, calibration_loader, test_loader,
                       dense_calib, dense_test, device):
    masked = copy.deepcopy(model).to(device)
    apply_masks(masked, masks)
    masked_calib = logits(masked, calibration_loader, device)
    masked_test = logits(masked, test_loader, device)
    calib_error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    test_error = (masked_test - dense_test).flatten(1).square().mean(dim=1)
    result = {
        'calib_distortion': float(calib_error.mean()),
        'test_distortion': float(test_error.mean()),
        'calib_max_distortion': float(calib_error.max()),
        'test_max_distortion': float(test_error.max()),
    }
    del masked, masked_calib, masked_test
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return result

def rank_correlation(predicted, actual):
    pred_rank = pd.Series(predicted).rank(method='average')
    actual_rank = pd.Series(actual).rank(method='average')
    value = pred_rank.corr(actual_rank)
    return float(value) if pd.notna(value) else float('nan')

def jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

In [ ]:
def run_seed(seed):
    started = time.monotonic()
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    run_root = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{seed}'
    checkpoint = run_root / 'dense_resnet32.pt'
    output_dir = OUTPUT_ROOT / f'seed_{seed}'
    output_dir.mkdir(parents=True, exist_ok=True)
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')

    loader_args = Namespace(
        synthetic=False,
        data_root=str(DATA_ROOT),
        download=False,
        train_samples=TRAIN_SAMPLES,
        calib_samples=CALIB_SAMPLES,
        test_samples=TEST_SAMPLES,
        batch_size=BATCH_SIZE,
        seed=seed,
    )
    (train_loader, calibration_loader, test_loader), split_info = make_loaders(loader_args)

    model = load_resnet32().to(device)
    checkpoint_sha256 = load_checkpoint(model, checkpoint)
    model.eval()
    layers = prunable_layers(model)
    if len(layers) != len(all_layer_names):
        raise RuntimeError('The checkpoint model has a different prunable-layer count.')

    dense_calib = logits(model, calibration_loader, device)
    dense_test = logits(model, test_loader, device)
    dense_accuracy = accuracy(model, test_loader, device)

    single = {}
    single_rows = []
    for layer_index in selected_layers:
        values = measure_distortion(
            model,
            masks_for_group(model, [layer_index]),
            calibration_loader,
            test_loader,
            dense_calib,
            dense_test,
            device,
        )
        single[layer_index] = values
        single_rows.append({
            'layer_index': int(layer_index),
            'layer_name': all_layer_names[layer_index],
            **values,
        })

    pair = {}
    pair_rows = []
    unique_pairs = sorted({
        tuple(sorted((int(a), int(b))))
        for group in candidate_groups
        for a in group
        for b in group
        if a < b
    })
    for layer_a, layer_b in unique_pairs:
        values = measure_distortion(
            model,
            masks_for_group(model, [layer_a, layer_b]),
            calibration_loader,
            test_loader,
            dense_calib,
            dense_test,
            device,
        )
        pair[(layer_a, layer_b)] = values
        pair_rows.append({
            'layer_a': layer_a,
            'layer_a_name': all_layer_names[layer_a],
            'layer_b': layer_b,
            'layer_b_name': all_layer_names[layer_b],
            **values,
        })

    group = {}
    group_rows = []
    for group_id, group_indices in enumerate(candidate_groups):
        values = measure_distortion(
            model,
            masks_for_group(model, group_indices),
            calibration_loader,
            test_loader,
            dense_calib,
            dense_test,
            device,
        )
        group[tuple(group_indices)] = values
        group_rows.append({
            'group_id': group_id,
            'layer_indices': ','.join(str(i) for i in group_indices),
            'layer_names': ' | '.join(all_layer_names[i] for i in group_indices),
            **values,
        })

    candidates = []
    for group_id, group_indices in enumerate(candidate_groups):
        additive_calib = sum(single[i]['calib_distortion'] for i in group_indices)
        additive_test = sum(single[i]['test_distortion'] for i in group_indices)
        pair_corrected_calib = additive_calib
        pair_corrected_test = additive_test
        for a_position, a in enumerate(group_indices):
            for b in group_indices[a_position + 1:]:
                pair_values = pair[tuple(sorted((a, b)))]
                pair_corrected_calib += (
                    pair_values['calib_distortion']
                    - single[a]['calib_distortion']
                    - single[b]['calib_distortion']
                )
                pair_corrected_test += (
                    pair_values['test_distortion']
                    - single[a]['test_distortion']
                    - single[b]['test_distortion']
                )
        joint_values = group[tuple(group_indices)]
        candidates.append({
            'seed': seed,
            'group_id': group_id,
            'layer_indices': ','.join(str(i) for i in group_indices),
            'layer_names': ' | '.join(all_layer_names[i] for i in group_indices),
            'additive_calib': additive_calib,
            'pair_corrected_calib': pair_corrected_calib,
            'joint_calib': joint_values['calib_distortion'],
            'additive_test': additive_test,
            'pair_corrected_test': pair_corrected_test,
            'joint_test': joint_values['test_distortion'],
            'joint_test_max': joint_values['test_max_distortion'],
        })

    candidates_df = pd.DataFrame(candidates)
    prediction_columns = ['additive_calib', 'pair_corrected_calib', 'joint_calib']
    prediction_actual_rows = []
    for column in prediction_columns:
        prediction_actual_rows.append({
            'seed': seed,
            'method': column.replace('_calib', ''),
            'rank_correlation_with_test_joint': rank_correlation(
                candidates_df[column],
                candidates_df['joint_test'],
            ),
            'calibration_mae_vs_test_joint': float(
                np.mean(np.abs(candidates_df[column] - candidates_df['joint_test']))
            ),
        })
    prediction_df = pd.DataFrame(prediction_actual_rows)

    test_oracle = float(candidates_df['joint_test'].min())
    selection_rows = []
    for method, score_column in {
        'additive': 'additive_calib',
        'pair_corrected': 'pair_corrected_calib',
        'direct_joint': 'joint_calib',
    }.items():
        selected_row = candidates_df.loc[candidates_df[score_column].idxmin()]
        group_indices = tuple(
            int(x) for x in selected_row['layer_indices'].split(',')
        )
        masks = masks_for_group(model, group_indices)
        masked_model = copy.deepcopy(model).to(device)
        apply_masks(masked_model, masks)
        immediate_accuracy = accuracy(masked_model, test_loader, device)
        finetuned_accuracy = immediate_accuracy
        if RUN_FINETUNE and FINETUNE_EPOCHS > 0:
            train_epochs(
                masked_model,
                train_loader,
                device,
                FINETUNE_EPOCHS,
                LEARNING_RATE,
                seed + 1000,
                masks=masks,
            )
            apply_masks(masked_model, masks)
            finetuned_accuracy = accuracy(masked_model, test_loader, device)

        test_distortion = float(selected_row['joint_test'])
        selected_score = float(selected_row[score_column])
        selection_rows.append({
            'seed': seed,
            'method': method,
            'group_id': int(selected_row['group_id']),
            'layer_indices': selected_row['layer_indices'],
            'layer_names': selected_row['layer_names'],
            'calibration_score': selected_score,
            'test_joint_distortion': test_distortion,
            'test_oracle_distortion': test_oracle,
            'relative_test_regret': (test_distortion - test_oracle) / max(test_oracle, 1e-12),
            'dense_accuracy': dense_accuracy,
            'immediate_accuracy': immediate_accuracy,
            'finetuned_accuracy': finetuned_accuracy,
            'retained_fraction': model_stats(masked_model)['retained_fraction'],
            'sparsity': model_stats(masked_model)['sparsity'],
        })
        torch.save(
            {'method': method, 'seed': seed, 'group_indices': group_indices, 'masks': masks},
            output_dir / f'{method}_masks.pt',
        )
        del masked_model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    candidates_df.to_csv(output_dir / 'candidate_scores.csv', index=False)
    pd.DataFrame(single_rows).to_csv(output_dir / 'single_layer_scores.csv', index=False)
    pd.DataFrame(pair_rows).to_csv(output_dir / 'pair_scores.csv', index=False)
    pd.DataFrame(group_rows).to_csv(output_dir / 'group_joint_scores.csv', index=False)
    prediction_df.to_csv(output_dir / 'prediction_fidelity.csv', index=False)
    selection_df = pd.DataFrame(selection_rows)
    selection_df.to_csv(output_dir / 'selection_accuracy.csv', index=False)

    best_group_by_test = candidates_df.loc[candidates_df['joint_test'].idxmin()]
    summary = {
        'seed': seed,
        'selection_seed': SELECTION_SEED,
        'device': str(device),
        'checkpoint': str(checkpoint),
        'checkpoint_sha256': checkpoint_sha256,
        'calibration_samples': CALIB_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'prune_ratio': PRUNE_RATIO,
        'finetune_epochs': FINETUNE_EPOCHS if RUN_FINETUNE else 0,
        'selected_layers': [
            {'index': int(i), 'name': all_layer_names[i]}
            for i in selected_layers
        ],
        'candidate_groups': [list(group) for group in candidate_groups],
        'dense_accuracy': dense_accuracy,
        'selection_results': selection_rows,
        'prediction_fidelity': prediction_actual_rows,
        'test_oracle_group_id': int(best_group_by_test['group_id']),
        'elapsed_seconds': time.monotonic() - started,
    }
    (output_dir / 'summary.json').write_text(
        json.dumps(summary, indent=2, default=jsonable) + '\n',
        encoding='utf-8',
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for method, column, color in [
        ('additive', 'additive_calib', 'tab:blue'),
        ('pair_corrected', 'pair_corrected_calib', 'tab:orange'),
        ('direct_joint', 'joint_calib', 'tab:green'),
    ]:
        axes[0].scatter(
            candidates_df[column],
            candidates_df['joint_test'],
            label=method,
            alpha=0.75,
            color=color,
        )
    limits = axes[0].get_xlim()
    axes[0].plot(limits, limits, 'k--', linewidth=1)
    axes[0].set_xlabel('Calibration prediction')
    axes[0].set_ylabel('Test joint distortion')
    axes[0].set_title(f'Seed {seed}: prediction fidelity')
    axes[0].legend()

    plot_df = selection_df.set_index('method')
    axes[1].bar(
        plot_df.index,
        plot_df['finetuned_accuracy'] if RUN_FINETUNE else plot_df['immediate_accuracy'],
        color=['tab:blue', 'tab:orange', 'tab:green'],
    )
    axes[1].axhline(dense_accuracy, color='black', linestyle='--', label='dense')
    axes[1].set_ylabel('Test accuracy (%)')
    axes[1].set_title(f'Seed {seed}: selected-mask accuracy')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / 'selection_fidelity.png', dpi=160)
    plt.close(fig)

    print(
        f'Seed {seed} finished in {time.monotonic() - started:.1f}s | '
        f'dense acc={dense_accuracy:.2f}% | '
        f'additive test regret={selection_rows[0]["relative_test_regret"]:.4f} | '
        f'pair-corrected test regret={selection_rows[1]["relative_test_regret"]:.4f}'
    )
    return selection_df, prediction_df, summary

all_selection = []
all_prediction = []
all_summaries = []
for seed in SEEDS:
    selection_df, prediction_df, summary = run_seed(seed)
    all_selection.append(selection_df)
    all_prediction.append(prediction_df)
    all_summaries.append(summary)

selection_results = pd.concat(all_selection, ignore_index=True)
prediction_results = pd.concat(all_prediction, ignore_index=True)
selection_results.to_csv(OUTPUT_ROOT / 'all_seeds_selection_accuracy.csv', index=False)
prediction_results.to_csv(OUTPUT_ROOT / 'all_seeds_prediction_fidelity.csv', index=False)

numeric_summary = selection_results.groupby('method').agg({
    'test_joint_distortion': ['mean', 'std'],
    'relative_test_regret': ['mean', 'std'],
    'immediate_accuracy': ['mean', 'std'],
    'finetuned_accuracy': ['mean', 'std'],
    'sparsity': ['mean', 'std'],
}).reset_index()
numeric_summary.to_csv(OUTPUT_ROOT / 'all_seeds_summary.csv', index=False)
(OUTPUT_ROOT / 'all_seeds_summary.json').write_text(
    json.dumps({
        'seeds': SEEDS,
        'selection_seed': SELECTION_SEED,
        'calibration_samples': CALIB_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'results': all_summaries,
    }, indent=2, default=jsonable) + '\n',
    encoding='utf-8',
)
display(numeric_summary)
print('Saved:', OUTPUT_ROOT)

## Publish results

This cell copies the new results into the experiment repository under `results/rdhp_rdp_selection_fidelity_all_seeds`.

In [ ]:
if not PUBLISH_RESULTS:
    print('Publishing disabled.')
else:
    from google.colab import userdata

    write_token = userdata.get('GITHUB_WRITE_TOKEN')
    if not write_token:
        raise RuntimeError('Add GITHUB_WRITE_TOKEN with Contents read/write access.')

    upload_repo = Path('/content/rdhp-rdp-selection-fidelity-upload')
    if upload_repo.exists():
        shutil.rmtree(upload_repo)

    askpass = Path('/tmp/github-askpass-selection-fidelity-write.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "  *Username*) printf '%s\\n' x-access-token ;;\n"
        "  *Password*) printf '%s\\n' \"$GITHUB_WRITE_TOKEN\" ;;\n"
        "  *) printf '\\n' ;;\n"
        'esac\n'
    )
    askpass.chmod(0o700)

    upload_env = os.environ.copy()
    upload_env.update({
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
        'GITHUB_WRITE_TOKEN': write_token,
    })

    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git',
            str(upload_repo),
        ], env=upload_env, check=True)

        destination = upload_repo / 'results' / 'rdhp_rdp_selection_fidelity_all_seeds'
        shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)

        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.name', 'Colab Results Bot'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.email', 'colab-results@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'add', 'results', '--'])
        changed = subprocess.run([
            'git', '-C', str(upload_repo), 'diff', '--cached', '--quiet'
        ]).returncode != 0

        if changed:
            subprocess.run([
                'git', '-C', str(upload_repo), 'commit', '-m',
                'Publish selection-fidelity results for seeds 42-44',
            ], check=True)
            subprocess.run([
                'git', '-C', str(upload_repo), 'push', 'origin', 'HEAD:master',
            ], env=upload_env, check=True)
            print('Selection-fidelity results uploaded.')
        else:
            print('No new results to upload.')
    finally:
        askpass.unlink(missing_ok=True)